## Вводные по кейсу


# Описание данных


## Таблица `Users`

| Поле              | Описание                                       |
|-------------------|------------------------------------------------|
| `user_id`         | Уникальный идентификатор пользователя.         |
| `registration_date` | Дата регистрации пользователя.                |
| `age`             | Возраст пользователя.                          |
| `gender`          | Пол.                                |
| `region`          | Регион.                                        |
| `acq_channel`     | Канал привлечения.                             |
| `buyer_segment`   | Сегмент покупателя.                            |
| `cohort_week`     | Неделя привлечения.|
| `cohort_month`    | Месяц привлечения. |

---


## Таблица `Events`

| Поле          | Описание                                                                 |
|---------------|--------------------------------------------------------------------------|
| `event_id`    | Уникальный идентификатор события.                                        |
| `user_id`     | Идентификатор пользователя.                                              |
| `event_date`  | Дата события.                                                            |
| `event_type`  | Тип события.   |
| `os`          | Операционная система.               |
| `device`      | Тип устройства.                                |
| `product_name`| Наименование товара, к которому относится событие (если применимо).      |
| `event_week`  | Неделя события.                                          |
| `event_month` | Месяц события .                                           |

---


## Таблица `Orders`

| Поле          | Описание                                                                 |
|---------------|--------------------------------------------------------------------------|
| `order_id`    | Уникальный идентификатор заказа.                                         |
| `user_id`     | Идентификатор пользователя, который сделал заказ |
| `order_date`  | Дата и время оформления заказа.                                          |
| `product_name`| Наименование товара.                                                     |
| `quantity`    | Количество единиц товара в заказе.                                       |
| `unit_price`  | Цена за одну единицу товара.                                             |
| `total_price` | Итоговая сумма заказа.                                                   |
| `category_name` | Наименование категории товара.                                         |
| `order_week`  | Неделя заказа.                                           |
| `order_month` | Месяц заказа.                                            |


## Таблица `Campaign_costs`

| Поле         | Описание                                                                 |
|--------------|--------------------------------------------------------------------------|
| `acq_channel`| Канал привлечения.  |
| `spend_month`| Месяц, в который был потрачен бюджет (отражает период, в котором были привлечены пользователи)                      |
| `budget`     | Маркетинговый бюджет (в денежном выражении), потраченный на данный канал в указанном месяце.                           |


## Описание событий

| Событие             | Описание                                                                                  |
|---------------------|-------------------------------------------------------------------------------------------|
| `page_view`         | Открытие любой страницы сайта или приложения пользователем.                                |
| `product_view`      | Просмотр страницы конкретного товара.                                                     |
| `product_click`     | Клик по товару (например, из списка товаров или на баннере).                              |
| `add_to_cart`       | Добавление товара в корзину.                                                              |
| `remove_from_cart`  | Удаление товара из корзины.                                                               |
| `search`            | Выполнение поиска по сайту или приложению.                                                |
| `filter_apply`      | Применение фильтра (например, по цене, бренду, категории).                                |
| `checkout_start`    | Начало оформления заказа (переход к оформлению корзины).                                  |
| `checkout_complete` | Завершение оформления заказа (не гарантирует покупку, покупка зависит от факта оплаты).                                          |
| `user_login`        | Вход пользователя в личный кабинет или аккаунт.                                           |
| `user_logout`       | Выход пользователя из аккаунта.                                                           |
| `wishlist_add`      | Добавление товара в список желаемого (wishlist).                                          |
| `category_view`     | Просмотр страницы категории товаров.                                                      |
| `banner_click`      | Клик по рекламному баннеру на сайте или в приложении.                                     |
| `promo_view`        | Просмотр промо-акции или специального предложения.                                        |


## Выручка маркетплейса = 5% от total_price


# Блок кода и расчета метрик


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind


### Предобработка данных


#### Таблица orders


In [ ]:
orders = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_orders.csv', parse_dates = [2,8,9])

orders=orders[orders['order_date']<'2025-01-01']
orders.head()

orders.info()


orders.duplicated().sum()


#### Таблица costs


In [ ]:
costs = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_campaign_costs.csv', index_col=0, parse_dates=[2])

costs = costs[costs['spend_month']<'2025-01-01']
costs.head()

costs.info()


costs.duplicated().sum()

duplicates = costs.groupby(['acq_channel', 'spend_month']).size().reset_index(name='count')
print(duplicates[duplicates['count'] > 1])

costs['acq_channel'].unique().tolist()


#### Таблица users


In [ ]:
users = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_users.csv', parse_dates=[1,7,8])

users=users[users['registration_date']<'2025-01-01']
users.head()

users.info()

users.duplicated().sum()


#### Таблица events


In [ ]:
events = pd.read_csv('https://code.s3.yandex.net/datasets/pa_marketplace_events.csv', parse_dates=[2,7,8])

events=events[events['event_date']<'2025-01-01']
events.head()

events.info()

events[events['product_name'].notna()]['event_type'].unique().tolist()

events.duplicated().sum()


Подводя итоги предобработки, данные достались в отличном состоянии - необъяснимых пропусков или дубликатов нет, форматы приведены


### 2. Построить обзор ключевых метрик


#### Revenue


In [ ]:
# Посчитаем выручку
total_revenue = orders['total_price'].sum()*0.05
# Сгруппируем для рассчета динамики по периодам: неделям и месяцам
revenue_week = orders.groupby('order_week')['total_price'].sum().reset_index(name='total_sum').sort_values('order_week')
revenue_month = orders.groupby('order_month')['total_price'].sum().reset_index(name='total_sum').sort_values('order_month')

revenue_week['revenue'] = revenue_week['total_sum']*0.05
revenue_month['revenue'] = revenue_month['total_sum']*0.05

# Скользящее среднее за 4 недели
revenue_week['rolling_4w'] = revenue_week['revenue'].rolling(window=4, min_periods=1).mean()
# Скользящее среднее за квартал
revenue_month['rolling_3m'] = revenue_month['revenue'].rolling(window=3, min_periods=1).mean()

print(f'Размер выручки за год в миллионах:{total_revenue/10**6:.2f}')

#Визуализируем динамику выручки по неделям и месяцам
def plot_func(x, y, y1, 
              marker='o', 
              label = 'Скользящее среднее', 
              alpha = 0.3, 
              title='',
              xlabel = 'Период',
              ylabel='Выручка'):
    plt.plot(x, y, marker=marker, label = 'Выручка')
    plt.plot(x, y1, label = label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(alpha=alpha)
    plt.legend()
    plt.tight_layout()
    plt.show

plt.figure(figsize=(10,6))
plot_func(x=revenue_week['order_week'],
         y=revenue_week['revenue'],
         y1=revenue_week['rolling_4w'],
         title='Выручка по неделям',
         xlabel='Неделя')

plt.figure(figsize=(10,6))
plot_func(x=revenue_month['order_month'],
         y=revenue_month['revenue'],
         y1=revenue_month['rolling_3m'],
         title='Выручка по месяцам',
         xlabel='Месяц')


**Размер годовой выручки: 22 миллиона**. Динамика по неделям и месяцам показывает быстрый рост в первой половине года - май становится самым успешным месяцем в году, за ним следует падение в июне. В целом летний квартал - самый успешный, вторая неделя июля показывает лучший результат в году, а июль и август возвращаются к цифрам мая. Но с началом сентября метрика быстро падает, последовательно, скаждым месяцем. В конце года достигается уровень начала марта.

*Динамика метрики выглядит удручающе* - можно было бы отметить сезонность как причину пика летом и падения осенью, но для маркетплейса сезонность, скорее всего, будет связана с праздниками и важными датами - тогда должен быть большой рост к черной пятнице и новому году.


#### ARPAU


In [ ]:
# Посчитаем количество активных пользователей по неделям и месяцам
active_users_week = events.groupby('event_week')['user_id'].nunique().reset_index(name='active_users')
active_users_month = events.groupby('event_month')['user_id'].nunique().reset_index(name='active_users')

# Присоединим выручку к полученным датафреймам
arpau_week = active_users_week.merge(revenue_week, how='left', left_on = 'event_week', right_on='order_week')
arpau_month = active_users_month.merge(revenue_month, how='left', left_on = 'event_month', right_on='order_month')

# Рассчитаем метрику
arpau_week['arpau'] = arpau_week['revenue']/arpau_week['active_users']
arpau_month['arpau'] = arpau_month['revenue']/arpau_month['active_users']

# Скользящее среднее за месяц и квартал
arpau_week['rolling_arpau_4w'] = arpau_week['arpau'].rolling(window=4,min_periods=1).mean()
arpau_month['rolling_arpau_3m'] = arpau_month['arpau'].rolling(window=3,min_periods=1).mean()

arpau_year = orders['total_price'].sum()*0.05/events['user_id'].nunique()

print(f'ARPAU за год в тысячах: {arpau_year/10**3:.2f}')

# Визуализируем результат
plt.figure(figsize=(10,6))
plot_func(x=arpau_week['event_week'],
         y=arpau_week['arpau'],
         y1=arpau_week['rolling_arpau_4w'],
         title='ARPAU по неделям',
         xlabel='Неделя')

plt.figure(figsize=(10,6))
plot_func(x=arpau_month['event_month'],
         y=arpau_month['arpau'],
         y1=arpau_month['rolling_arpau_3m'],
         title='ARPAU по месяцам',
         xlabel='Месяц')


**ARPAU за год: 0.99 тысяч.** Динамика по неделям показывает нестабильность метрики - за неделей подъема часто следует спад. В целом динамика метрики похожа на динамику выручки - стабильный рост в первой половине года, падение в июле и до конца года. Показатели во второй половине года значительно ухудшаются


#### ARPPU


In [ ]:
# Посчитаем количество платящих пользователей по неделям и месяцам
arppu_week = (orders.groupby('order_week').agg(
    paying_users=('user_id','nunique'),
    total_sum=('total_price','sum'))
    .reset_index()
)
arppu_month = (orders.groupby('order_month').agg(
    paying_users=('user_id','nunique'),
    total_sum=('total_price','sum'))
    .reset_index()
)

arppu_week['revenue'] = arppu_week['total_sum']*0.05
arppu_month['revenue'] = arppu_month['total_sum']*0.05

# Рассчитаем метрику
arppu_week['arppu'] = arppu_week['revenue']/arppu_week['paying_users']
arppu_month['arppu'] = arppu_month['revenue']/arppu_month['paying_users']

# Скользящее среднее за 4 недели и квартал
arppu_week['rolling_arppu_4w'] = arppu_week['arppu'].rolling(window=4,min_periods=1).mean()
arppu_month['rolling_arppu_3m'] = arppu_month['arppu'].rolling(window=3,min_periods=1).mean()

print(f"Среднее ARPPU за год: {orders['total_price'].sum()*0.05/orders['user_id'].nunique():.2f}")

# Визуализируем результат
plt.figure(figsize=(10,6))
plot_func(x=arppu_week['order_week'],
         y=arppu_week['arppu'],
         y1=arppu_week['rolling_arppu_4w'],
         title='ARPPU по неделям',
         xlabel='Неделя')

plt.figure(figsize=(10,6))
plot_func(x=arppu_month['order_month'],
         y=arppu_month['arppu'],
         y1=arppu_month['rolling_arppu_3m'],
         title='ARPPU по месяцам',
         xlabel='Месяц')


Метрика последовательно растет до августа, с сентября выходя на плато, **общегодовой показатель - 2465**. Важная особенность метрики - высокая волатильность: метрика от недели к неделе может меняться  на 15-20%. Ее динамика не похожа на ARPAU - вероятно, существуют проблемы с конверсией активных пользователей в заказ.


#### WAU и MAU


In [ ]:
# Визуализируем WAU и MAU, они уже посчитаны 
# Таблицы active_users_week и active_users_month
active_users_week['rolling_wau_4w'] = active_users_week['active_users'].rolling(window=4,min_periods=1).mean()
active_users_month['rolling_mau_3m'] = active_users_month['active_users'].rolling(window=3,min_periods=1).mean()

print(f"Всего акивных пользователей за год: {events['user_id'].nunique()}")

plt.figure(figsize=(10,6))
plot_func(x=active_users_week['event_week'],
         y=active_users_week['active_users'],
         y1=active_users_week['rolling_wau_4w'],
         title='Динамика WAU',
         xlabel='Неделя',
         ylabel='Количество активных пользователей')

plt.figure(figsize=(10,6))
plot_func(x=active_users_month['event_month'],
         y=active_users_month['active_users'],
         y1=active_users_month['rolling_mau_3m'],
         title='Динамика MAU',
         xlabel='Месяц',
         ylabel='Количество активных пользователей')


Динамика метрик по месяцам и неделям показывает стабильный рост, что, после небольшого спада в июле, продолжается до августа. Начиная с осени, WAU и MAU выходят на плато - не показывают роста, но достаточно медленно падают. 

Активность пользователей и ARPPU не соответсвуют такому же падению, как и ARPAU, Revenue, т.е. падение общей выручки не обязательно вызвано сокращением пользовательской активности. Вероятные причины - Снижение конверсии активных пользователей в заказ, также можно проверить динамику среднего чека.


#### AOV 


In [ ]:
# Рассчитаем AOV по неделям и месяцам
aov_week = orders.groupby('order_week')['total_price'].mean().reset_index(name='AOV')
aov_month = orders.groupby('order_month')['total_price'].mean().reset_index(name='AOV')

# Скользящее среднее за 4 недели и квартал
aov_week['rolling_aov_4w'] = aov_week['AOV'].rolling(window=4,min_periods=1).mean()
aov_month['rolling_aov_3m'] = aov_month['AOV'].rolling(window=3,min_periods=1).mean()

print(f"AOV за год: {orders['total_price'].mean():.2f}")

# Визуализируем результат
plt.figure(figsize=(10,6))
plot_func(x=aov_week['order_week'],
         y=aov_week['AOV'],
         y1=aov_week['rolling_aov_4w'],
         title='AOV по неделям',
         xlabel='Неделя',
         ylabel='AOV')

plt.figure(figsize=(10,6))
plot_func(x=aov_month['order_month'],
         y=aov_month['AOV'],
         y1=aov_month['rolling_aov_3m'],
         title='AOV по месяцам',
         xlabel='Месяц',
         ylabel='AOV')


Динамика метрики показывает стабильный, последовательный рост. Исключением стали сентябрь и октябрь, также большой провал в декабре, но последняя неделя года стала самой успешной. 


#### CR из активных пользователей в платящих


In [ ]:
# Расчитаем WPU и MPU
wpu = orders.groupby('order_week')['user_id'].nunique().reset_index(name='wpu')
mpu = orders.groupby('order_month')['user_id'].nunique().reset_index(name='mpu')

# Присоединим получаенные данные к таблицам ARPAU
conversion_week = active_users_week.merge(wpu, how='left', left_on='event_week',right_on='order_week')
conversion_month = active_users_month.merge(mpu, how='left', left_on='event_month',right_on='order_month')

# Расчитаем метркиу
conversion_week['cr'] = (conversion_week['wpu']/conversion_week['active_users']*100).round(2)
conversion_month['cr'] = (conversion_month['mpu']/conversion_month['active_users']*100).round(2)

# Скользящее среднее за 4 недели и квартал
conversion_week['rolling_cr_4w'] = conversion_week['cr'].rolling(window=4,min_periods=1).mean()
conversion_month['rolling_cr_3m'] = conversion_month['cr'].rolling(window=3,min_periods=1).mean()

# Визуализируем результат
plt.figure(figsize=(10,6))
plot_func(x=conversion_week['event_week'],
         y=conversion_week['cr'],
         y1=conversion_week['rolling_cr_4w'],
         title='cr по неделям',
         xlabel='Неделя',
         ylabel='cr')

plt.figure(figsize=(10,6))
plot_func(x=conversion_month['event_month'],
         y=conversion_month['cr'],
         y1=conversion_month['rolling_cr_3m'],
         title='cr по месяцам',
         xlabel='Месяц',
         ylabel='cr')


Обнаруживаем проблему - конверсия начинает падать уже с мая, единственное исключение - недели июня. И падение это стремительное  - метрика падает с 40% в пиках по месяцам до 24%, с 34% в недельных пиках до менее чем 20% в декабре.


### 3. Оценка метрик монетизации и юнит-экономики


#### CAC


In [ ]:
# Посчитаем уникальных пользователей по месяцам
users_monthly = users.groupby('cohort_month')['user_id'].nunique().reset_index(name='users')

# Посчитаем общий бюджет по месяцам
budget_monthly = costs.groupby('spend_month')['budget'].sum().reset_index().rename(columns={'spend_month': 'cohort_month'})

# Объединим данные и рассчитаем метрику
cac_monthly = users_monthly.merge(budget_monthly, how='left', on='cohort_month')
cac_monthly['cac'] = (cac_monthly['budget']/cac_monthly['users']).round(2)

cac_monthly['rolling_cac_3m'] = (cac_monthly['cac'].rolling(window=3,min_periods=1).mean()).round(2)

print(f"Средний CAC за год:{costs['budget'].sum()/users['user_id'].nunique():.2f}")

plt.figure(figsize=(10,6))
plot_func(x=cac_monthly['cohort_month'],
         y=cac_monthly['cac'],
         y1=cac_monthly['rolling_cac_3m'],
         title='Динамика CAC',
         xlabel='Месяц',
         ylabel='CAC')


График показывает стабильный рост CAC до июля - метрика достигает пикового значения 407.5. После июля она медленно падает - достигая 403 в декабре.


#### ARPU


In [ ]:
# Объединим даныне и посчитаем метркиу
orders_with_cohort = orders.merge(users[['user_id', 'cohort_month']], how='left', on='user_id')
# Добавим месяц жизни
orders_with_cohort['month_life'] = orders_with_cohort['order_month'].dt.month - orders_with_cohort['cohort_month'].dt.month+1
# Сгруппируем по когорте и месяцу жизни
cohort_df = orders_with_cohort.groupby(['cohort_month', 'month_life']).agg(
    users = ('user_id', 'nunique'),
    total_sum = ('total_price', 'sum')).reset_index()
    
cohort_df['revenue'] = cohort_df['total_sum']*0.05

# Добавим размер когорты
cohort_size = users.groupby('cohort_month')['user_id'].nunique().reset_index(name='cohort_users')
cohort_df = cohort_df.merge(cohort_size, how='left', on='cohort_month')

cohort_df['arpu'] = (cohort_df['revenue'] / cohort_df['cohort_users']).round(2)

pivot_arpu = cohort_df.pivot(index='cohort_month', columns='month_life', values='arpu')
pivot_arpu.index = pivot_arpu.index.strftime('%Y-%m')

plt.figure(figsize=(12, 8))
sns.heatmap(pivot_arpu, annot=True, fmt='.1f', cmap='coolwarm')
plt.title('ARPU по месяцам жизни когорт')
plt.xlabel('Месяц жизни')
plt.ylabel('Когорта')
plt.tight_layout()
plt.show()


Тепловая карта показывает, что для горот самым успешным месяцем становится второй - он всегда лучше первого. Самые успешные когорты - мартовская и апрельская, у них спад ARPU самый медленный. Главное падение в ARPU происходит начиная с августа. Самым провальным же месяцем для каждой когорты становится третий - у всех его значения падают в два раза по сравнению со вторым.


#### ROI


In [ ]:
# Расчитаем LTV
cohort_df['ltv'] = cohort_df.groupby('cohort_month')['arpu'].cumsum().round(2)
# Добавим CAC по когорте
cohort_df = cohort_df.merge(cac_monthly[['cohort_month', 'cac']], how='left', on='cohort_month')

# Расчитаем ROI
cohort_df['roi'] = (cohort_df['ltv']-cohort_df['cac'])/cohort_df['cac']*100

pivot_roi = cohort_df.pivot(index='cohort_month', columns='month_life', values='roi')
pivot_roi.index = pivot_roi.index.strftime('%Y-%m')

plt.figure(figsize=(12, 8))
sns.heatmap(pivot_roi, annot=True, fmt='.1f', cmap='coolwarm')
plt.title('ROI по месяцам жизни когорт')
plt.xlabel('Месяц жизни')
plt.ylabel('Когорта')
plt.tight_layout()
plt.show()


Обнаруживаем, что все когорты до июля уходят в окупаемость уже на второй месяц. Самые успешные когорты: мартовская и апрельская, здесь ROI уходит за 100. Последние 4 когорты не показывают положительного ROI. Еще одно замечание - резкое падение роста ROI с 5 месяца.


#### Payback period


In [ ]:
# Расчитаем payback_period с помощью ROI
payback = cohort_df[cohort_df['roi']>=0].groupby('cohort_month')['month_life'].min().reset_index(name='payback_period')

# Сортируем по дате
payback = payback.sort_values('cohort_month')

plt.figure(figsize=(10, 6))
plt.plot(payback['cohort_month'], payback['payback_period'], marker='o')
plt.title('Динамика периода окупаемости по когортам')
plt.xlabel('Когорта')
plt.ylabel('Payback period')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


График показывает, что все когорты до июля окупались за два месяца - июльская уже за 3, августовская окупилась только к концу года, а оставшиеся не окупились совсем. Важно, что у них также значительно ниже ROI, значит что даже при росте arpu в новом году окупаться они все равно будут дольше.


## Выводы по этапам 2 и 3



За рассматриваемый год маркетплейс получил около **22 млн выручки**, однако рост продукта оказался неустойчивым. Основное увеличение показателей пришлось на первую половину года и летний период. *Начиная с сентября выручка, ARPAU, ARPU и ROI последовательно снижаются*, а ожидаемого роста в период Чёрной пятницы и новогодних праздников не наблюдается. Поэтому объяснять ухудшение результатов только сезонностью нельзя.

*Главной точкой снижения выглядит конверсия активных пользователей в покупателей*. WAU и MAU осенью выходят на плато и сокращаются сравнительно медленно, средний чек в целом растёт, а ARPPU остаётся высоким. Следовательно, падение выручки связано не столько с уменьшением аудитории или расходов платящих пользователей, сколько с сокращением доли активных пользователей, совершающих заказ. **Это подтверждается снижением конверсии примерно с 40% до 24% по месяцам и менее чем до 20% по неделям**.

Таким образом, продукт сохраняет активную аудиторию и способен получать значительную выручку от платящих пользователей, но всё хуже переводит активность в покупки. Возможные причины следует искать в качестве привлекаемого трафика, изменениях воронки оформления заказа, ассортименте, ценах, промоакциях, условиях доставки и технических ошибках. Отдельно стоит сравнить конверсию по каналам привлечения, устройствам, категориям товаров, а также новым и повторным пользователям.

Показатели юнит-экономики также указывают на риски. При среднем **CAC 437,75 и среднем месячном ARPU 41,52** привлечение пользователя окупается примерно за **10,5 месяца**. Такой срок близок к году, поэтому его приемлемость зависит от того, сохраняет ли пользователь активность на протяжении всего периода. *Переход месячного ROI в отрицательную область во второй половине года означает, что масштабирование маркетинговых расходов без улучшения конверсии может привести к росту убытков.*

При этом рост ARPPU и среднего чека показывает, что проблема находится преимущественно не в монетизации уже платящих клиентов. Основным направлением развития должно стать увеличение числа активных пользователей, совершающих заказ, и повышение частоты повторных покупок. Это позволит использовать существующую аудиторию эффективнее, сократить срок окупаемости и восстановить рост выручки.


## 4. Поиск инсайтов, точек роста и сегментация


### Сегментация


#### Сегментация по `buyer_segment`


In [ ]:
# Добавим к events и orders информацию о сегменте покупателя
# Добавляем сегмент через map
segment_map = users.set_index('user_id')['buyer_segment']
events['buyer_segment'] = events['user_id'].map(segment_map).fillna('unknown')
orders['buyer_segment'] = orders['user_id'].map(segment_map).fillna('unknown')

# Посчитаем количество активных пользователей по неделям и месяцам с учетом сегментов
active_users_week_segments = (events.groupby(['event_week', 'buyer_segment'])['user_id']
                              .nunique()
                              .reset_index(name='active_users'))
active_users_month_segments = (events.groupby(['event_month', 'buyer_segment'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))

# Посчитаем выручку по неделям и месяцам с учетом сегментов
revenue_week_segments = (orders.groupby(['order_week', 'buyer_segment'])['total_price']
                         .sum()
                         .reset_index(name='total_sum'))
revenue_month_segments = (orders.groupby(['order_month', 'buyer_segment'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))

revenue_week_segments['revenue'] = revenue_week_segments['total_sum'] * 0.05
revenue_month_segments['revenue'] = revenue_month_segments['total_sum'] * 0.05

# Присоединим выручку к полученным датафреймам
arpau_week_segments = active_users_week_segments.merge(
    revenue_week_segments,
    how='left',
    left_on=['event_week', 'buyer_segment'],
    right_on=['order_week', 'buyer_segment']
)
arpau_month_segments = active_users_month_segments.merge(
    revenue_month_segments,
    how='left',
    left_on=['event_month', 'buyer_segment'],
    right_on=['order_month', 'buyer_segment']
)

# Посчитаем ARPAU
arpau_week_segments['arpau'] = (arpau_week_segments['revenue'] / arpau_week_segments['active_users']).fillna(0)
arpau_month_segments['arpau'] = (arpau_month_segments['revenue'] / arpau_month_segments['active_users']).fillna(0)

# Сгруппируем данные для расчета конверсии
# Расчитаем WPU и MPU
wpu_segments = orders.groupby(['order_week', 'buyer_segment'])['user_id'].nunique().reset_index(name='wpu')
mpu_segments = orders.groupby(['order_month', 'buyer_segment'])['user_id'].nunique().reset_index(name='mpu')

# Присоединим получаенные данные к таблицам ARPAU
conversion_week_segments = active_users_week_segments.merge(
    wpu_segments,
    how='left',
    left_on=['event_week', 'buyer_segment'],
    right_on=['order_week', 'buyer_segment'])
conversion_month_segments = active_users_month_segments.merge(
    mpu_segments,
    how='left',
    left_on=['event_month', 'buyer_segment'],
    right_on=['order_month', 'buyer_segment'])

# Расчёт конверсии
conversion_week_segments['cr'] = (conversion_week_segments['wpu'] / conversion_week_segments['active_users'] * 100).fillna(0).round(2)
conversion_month_segments['cr'] = (conversion_month_segments['mpu'] / conversion_month_segments['active_users'] * 100).fillna(0).round(2)


# Визуализируем метрики по сегментам
plt.figure(figsize=(8,6))
for segment in arpau_week_segments['buyer_segment'].unique():
    df_plot = arpau_week_segments[arpau_week_segments['buyer_segment'] == segment].sort_values('event_week')
    plt.plot(df_plot['event_week'], df_plot['active_users'], marker='o', label=segment)

plt.title('WAU по сегментам')
plt.xlabel('Дата')
plt.ylabel('WAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
for segment in arpau_month_segments['buyer_segment'].unique():
    df_plot = arpau_month_segments[arpau_month_segments['buyer_segment'] == segment].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['active_users'], marker='o', label=segment)

plt.title('MAU по сегментам')
plt.xlabel('Дата')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика сегментов отражает общую динамику WAU и MAU. Самый большой сегмент покупателей - medium, в пике их количество в месяц достигает 1600. Следующий сегмент  - regular, в некоторые недели догоняет medium, но в целом значительно отстает. Третий сегмент - rare - незначительно уступает regular. Хуже всего себя показывают first_time и unknown, что объяснимо.


In [ ]:
plt.figure(figsize=(8,6))
for segment in arpau_week_segments['buyer_segment'].unique():
    df_plot = arpau_week_segments[arpau_week_segments['buyer_segment'] == segment].sort_values('event_week')
    plt.plot(df_plot['event_week'], df_plot['revenue'], marker='o', label=segment)

plt.title('Динамика выручки по сегментам')
plt.xlabel('Дата')
plt.ylabel('revenue')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
for segment in arpau_month_segments['buyer_segment'].unique():
    df_plot = arpau_month_segments[arpau_month_segments['buyer_segment'] == segment].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['revenue'], marker='o', label=segment)

plt.title('Динамика выручки по сегментам')
plt.xlabel('Дата')
plt.ylabel('Revenue')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Лучше всего по выручке себя проявляет сегмент regular - он достигает пиковых значений в мае, августе, поле чего падает - его динамика похожа на общую динамику выручки компании. Сегмент medium здесь значительно уступает regular, имея похожую динамику. Оставшиеся три сегмента имеют минимальные размеры выручки. Можно отметить, что ключевыми клиентами становятся regular, которые не так активно заходят в приложение, но при этом приносят большуюв выручку.


In [ ]:
plt.figure(figsize=(8,6))
for segment in arpau_week_segments['buyer_segment'].unique():
    df_plot = arpau_week_segments[arpau_week_segments['buyer_segment'] == segment].sort_values('event_week')
    plt.plot(df_plot['event_week'], df_plot['arpau'], marker='o', label=segment)

plt.title('Динамика arpau по сегментам')
plt.xlabel('Дата')
plt.ylabel('arpau')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
for segment in arpau_month_segments['buyer_segment'].unique():
    df_plot = arpau_month_segments[arpau_month_segments['buyer_segment'] == segment].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['arpau'], marker='o', label=segment)

plt.title('Динамика arpau по сегментам')
plt.xlabel('Дата')
plt.ylabel('arpau')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


График показывает, что ARPAU нашей ключевой группы - regular начинает падать во второй половине года, его догоняет сегмент unknown - не отмеченные пользователи. Третьим отмечаем medium - он также ухудшается во второй половине года, как и regular. Худшими стали rare и one_time. К концу года выручка на rare стала даже ниже чем у one_time.


In [ ]:
plt.figure(figsize=(8,6))
for segment in conversion_week_segments['buyer_segment'].unique():
    df_plot = conversion_week_segments[conversion_week_segments['buyer_segment'] == segment].sort_values('event_week')
    plt.plot(df_plot['event_week'], df_plot['cr'], marker='o', label=segment)

plt.title('Динамика cr по сегментам')
plt.xlabel('Дата')
plt.ylabel('cr')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
for segment in conversion_month_segments['buyer_segment'].unique():
    df_plot = conversion_month_segments[conversion_month_segments['buyer_segment'] == segment].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['cr'], marker='o', label=segment)

plt.title('Динамика cr по сегментам')
plt.xlabel('Дата')
plt.ylabel('cr')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика всех сегментов, кроме unknown показывает снижение - показатель того, что снижение конверсии, а за ней и выручки - общая проблема для всех пользователей, независимо от их сегмента


#### Сегментация по `acq_channel`


In [ ]:
# Расчитаем CAC по сегментам
users_monthly_channel = users.groupby(['acq_channel','cohort_month'])['user_id'].nunique().reset_index(name='users')
costs_monthly_channel = (costs.groupby(['acq_channel', 'spend_month'])['budget']
                         .sum()
                         .reset_index()
                         .rename(columns={'spend_month': 'cohort_month'})
                        )

cac_monthly_channel = users_monthly_channel.merge(costs_monthly_channel,
                                                 how='left',
                                                 on=['acq_channel', 'cohort_month'])
# Рассчитываем CAC
cac_monthly_channel['cac'] = (cac_monthly_channel['budget'] / cac_monthly_channel['users']).round(2)

plt.figure(figsize=(8,6))
for channel in cac_monthly_channel['acq_channel'].unique():
    df_plot = cac_monthly_channel[cac_monthly_channel['acq_channel'] == channel].sort_values('cohort_month')
    plt.plot(df_plot['cohort_month'], df_plot['cac'], marker='o', label=channel)

plt.title('Динамика cac по каналам')
plt.axhline(cac_monthly['cac'].mean(), linestyle='--')
plt.xlabel('Дата')
plt.ylabel('cac')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика CAC по каналам показывает стабильность для каждого из каналов привлечения. Самый дорогие - Google Ads и TikTok. Самые дешевые - Email Marketing и SEO.


In [ ]:
# Добавим к events и orders информацию о канале покупателя
# Добавляем канал через map
channel_map = users.set_index('user_id')['acq_channel']
events['acq_channel'] = events['user_id'].map(channel_map).fillna('unknown')
orders['acq_channel'] = orders['user_id'].map(channel_map).fillna('unknown')

# Посчитаем количество активных пользователей по месяцам с учетом каналов
active_users_month_channel = (events.groupby(['event_month', 'acq_channel'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))

# Посчитаем выручку по месяцам с учетом каналов
revenue_month_channel = (orders.groupby(['order_month', 'acq_channel'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))

revenue_month_channel['revenue'] = revenue_month_channel['total_sum'] * 0.05

# Присоединим выручку к полученному датафрейму
arpau_month_channel = active_users_month_channel.merge(
    revenue_month_channel,
    how='left',
    left_on=['event_month', 'acq_channel'],
    right_on=['order_month', 'acq_channel'])

# Посчитаем ARPAU
arpau_month_channel['arpau'] = (arpau_month_channel['revenue'] / arpau_month_channel['active_users']).fillna(0)

# Сгруппируем данные для расчета конверсии
# Расчитаем MPU
mpu_channel = orders.groupby(['order_month', 'acq_channel'])['user_id'].nunique().reset_index(name='mpu')

# Присоединим получаенные данные к таблицам ARPAU
conversion_month_channel = active_users_month_channel.merge(
    mpu_channel,
    how='left',
    left_on=['event_month', 'acq_channel'],
    right_on=['order_month', 'acq_channel'])

# Расчёт конверсии
conversion_month_channel['cr'] = (conversion_month_channel['mpu'] / conversion_month_channel['active_users'] * 100).fillna(0).round(2)


plt.figure(figsize=(8,6))
for channel in arpau_month_channel['acq_channel'].unique():
    df_plot = arpau_month_channel[arpau_month_channel['acq_channel'] == channel].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['active_users'], marker='o', label=channel)

plt.title('MAU по каналам')
plt.xlabel('Дата')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


До августа самым популярным каналом привлечения был google ads, но с сентября его перегоняет и стремительно уходит вперед TikTok. Именно эти два канала были самыми дорогими, падение google ads на предпоследнее место к концу города говорит о явной проблеме с привлечением пользователей через этот канал. Третьим по популярности весь год оставался affiliate. SEO и Social media боролись за 4 место. Для SEO это хороший показатель, учитывая что он с отрывом самый недорогой.


In [ ]:
plt.figure(figsize=(8,6))
for channel in arpau_month_channel['acq_channel'].unique():
    df_plot = arpau_month_channel[arpau_month_channel['acq_channel'] == channel].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['revenue'], marker='o', label=channel)

plt.title('Выручка по каналам')
plt.xlabel('Дата')
plt.ylabel('revenue')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Самую большую выручку до октября приносили пользователи с Google ads, но его обгоняют юзеры с канала affiliate. Рост TikTok здесь также обнаруживается начиная с июля, но ему не удается подняться в тройку. В то же время, Google ads и здесь падает почти на дно, занимая третью с конца строчку в конце года. SEO все также показывает неплохой результат, долгое время идя четвертым, но к концу года его обгоняет TikTok.


In [ ]:
plt.figure(figsize=(8,6))
for channel in arpau_month_channel['acq_channel'].unique():
    df_plot = arpau_month_channel[arpau_month_channel['acq_channel'] == channel].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['arpau'], marker='o', label=channel)

plt.title('ARPAU по каналам')
plt.xlabel('Дата')
plt.ylabel('ARPAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Лучший результат по ARPAU показывают пользователи с канала affiliate, на втором месте идет social media, но к концу года его обгоняет google ads. Взлет TikTok незаметен - рост числа пользователей явно не дал достаточно роста выручки.


In [ ]:
plt.figure(figsize=(8,6))
for channel in conversion_month_channel['acq_channel'].unique():
    df_plot = conversion_month_channel[conversion_month_channel['acq_channel'] == channel].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['cr'], marker='o', label=channel)

plt.title('CR по каналам')
plt.xlabel('Дата')
plt.ylabel('CR')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Лучшим каналом по конверсии в покупку оказывается affiliate, следом идут social media и google_ads - его провалв активности пользователей не повлиял на их конверсию. Хуже всех себя показывает TikTok - большую часть времени конверсия отсюда даже не достигает 10%


#### Сегментация по `category_name`


In [ ]:
# Расчитаем топ-5 категорий
categories = orders.groupby('category_name')['order_id'].nunique().reset_index(name='orders')
categories = categories.sort_values('orders', ascending=False).head(5)

# Получаем список названий топ-5 категорий
top5_categories = categories['category_name'].tolist()

# Фильтруем заказы только по этим категориям
top5_orders = orders[orders['category_name'].isin(top5_categories)]

# Динамика по месяцам
dynamic = top5_orders.groupby(['order_month', 'category_name'])['order_id'].nunique().reset_index(name='orders')

# Построение графика
plt.figure(figsize=(10,6))
for category in top5_categories:
    df_plot = dynamic[dynamic['category_name'] == category].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['orders'], marker='o', label=category)

plt.title('Динамика 5 самых популярных категорий')
plt.xlabel('Месяц')
plt.ylabel('Количество заказов')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


С большим отрывом самая частая категория - Продукты питания. Можно отметить, что ее динамика схожа с общей днамикой выручки компании.


In [ ]:
# Топ-5 категорий по среднему AOV за весь период
avg_aov_per_category = orders.groupby('category_name')['total_price'].mean().sort_values(ascending=False)
top5_categories_aov = avg_aov_per_category.head(5).index.tolist()

# Фильтруем заказы по этим категориям и считаем AOV по месяцам
aov_top5 = orders[orders['category_name'].isin(top5_categories_aov)]
aov_monthly = aov_top5.groupby(
    ['order_month', 'category_name']
)['total_price'].mean().reset_index(name='AOV')

# График
plt.figure(figsize=(8,6))
for category in top5_categories_aov:
    df_plot = aov_monthly[aov_monthly['category_name'] == category].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['AOV'], marker='o', label=category)

plt.title('Динамика AOV по 5 самым прибыльным категориям')
plt.xlabel('Месяц')
plt.ylabel('Средний чек (AOV)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Самый высокий AOV у категории: мебель для дома, здесь средний чек превышает 70 тысяч. За ним следуют Бытовая техника и Украшения и часы. Провалов метрики у категорий не обнаружено


#### Сегментация по `device`


In [ ]:
# Добавляем девайс через map
device_map = events.sort_values('event_date').groupby('user_id')['device'].first()
orders['device'] = orders['user_id'].map(device_map).fillna('unknown')

# Активные пользователи по месяцам и устройствам
active_users_month_device = (events.groupby(['event_month', 'device'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))

# Выручка по месяцам и устройствам
revenue_month_device = (orders.groupby(['order_month', 'device'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))
revenue_month_device['revenue'] = revenue_month_device['total_sum'] * 0.05

# ARPAU
arpau_month_device = active_users_month_device.merge(
    revenue_month_device,
    how='left',
    left_on=['event_month', 'device'],
    right_on=['order_month', 'device']
)
# Расчитываем ARPAU
arpau_month_device['arpau'] = (arpau_month_device['revenue'] / arpau_month_device['active_users']).fillna(0)

# MPU
mpu_device = orders.groupby(['order_month', 'device'])['user_id'].nunique().reset_index(name='mpu')

# Конверсия
conversion_month_device = active_users_month_device.merge(
    mpu_device,
    how='left',
    left_on=['event_month', 'device'],
    right_on=['order_month', 'device']
)
conversion_month_device['cr'] = (conversion_month_device['mpu'] / 
                                 conversion_month_device['active_users'] * 100).fillna(0).round(2)

plt.figure(figsize=(8,6))
for device in arpau_month_device['device'].unique():
    df_plot = arpau_month_device[arpau_month_device['device'] == device].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['active_users'], marker='o', label=device)

plt.title('MAU по девайсам')
plt.xlabel('Месяц')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Все девайсы имеют схожую динамику, но выходят на пики в разное время, планшеты достигают вершины в мае, а mobile и desktop в августе-сентябре.


In [ ]:
plt.figure(figsize=(8,6))
for device in arpau_month_device['device'].unique():
    df_plot = arpau_month_device[arpau_month_device['device'] == device].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['revenue'], marker='o', label=device)

plt.title('Выручка по девайсам')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика выручки также схожа на всех устройствах, однако планшеты начинают падать на месяц раньше, чем дектопы и мобильные устройства


In [ ]:
plt.figure(figsize=(8,6))
for device in arpau_month_device['device'].unique():
    df_plot = arpau_month_device[arpau_month_device['device'] == device].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['arpau'], marker='o', label=device)

plt.title('ARPAU по девайсам')
plt.xlabel('Месяц')
plt.ylabel('ARPAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


ARPAU показывает, что пиковые значения достигаются на мобильных устройствах в мае, а на планшете и десктопе раньше. Однако с июля начинается скорое падение и в итоге мобильные устройства вовсе оказываются на последнем месте в конце года.


In [ ]:
plt.figure(figsize=(8,6))
for channel in conversion_month_device['device'].unique():
    df_plot = conversion_month_device[conversion_month_device['device'] == channel].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['cr'], marker='o', label=channel)

plt.title('CR по девайсам')
plt.xlabel('Дата')
plt.ylabel('CR')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Конверсия на мобильных устройствах начинает падать уже в мае, а на остальных девайсах с июля. В итоге конверсия падает с 35% до менее чем 20%


#### Сегментация по `os`


In [ ]:
# Добавляем ос через map
os_map = events.sort_values('event_date').groupby('user_id')['os'].first()
orders['os'] = orders['user_id'].map(os_map).fillna('unknown')

# Активные пользователи по месяцам и ос
active_users_month_os = (events.groupby(['event_month', 'os'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))

# Выручка по месяцам и ос
revenue_month_os = (orders.groupby(['order_month', 'os'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))
revenue_month_os['revenue'] = revenue_month_os['total_sum'] * 0.05

# ARPAU
arpau_month_os = active_users_month_os.merge(
    revenue_month_os,
    how='left',
    left_on=['event_month', 'os'],
    right_on=['order_month', 'os']
)
# Расчитываем ARPAU
arpau_month_os['arpau'] = (arpau_month_os['revenue'] / arpau_month_os['active_users']).fillna(0)

# MPU
mpu_os = orders.groupby(['order_month', 'os'])['user_id'].nunique().reset_index(name='mpu')

# Конверсия
conversion_month_os = active_users_month_os.merge(
    mpu_os,
    how='left',
    left_on=['event_month', 'os'],
    right_on=['order_month', 'os']
)
conversion_month_os['cr'] = (conversion_month_os['mpu'] / 
                                 conversion_month_os['active_users'] * 100).fillna(0).round(2)

plt.figure(figsize=(8,6))
for os in arpau_month_os['os'].unique():
    df_plot = arpau_month_os[arpau_month_os['os'] == os].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['active_users'], marker='o', label=os)

plt.title('MAU по ос')
plt.xlabel('Месяц')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика всех ос очень схожая, больше всех выделяется android, который обладает большей волатильностью.


In [ ]:
plt.figure(figsize=(8,6))
for os in arpau_month_os['os'].unique():
    df_plot = arpau_month_os[arpau_month_os['os'] == os].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['revenue'], marker='o', label=os)

plt.title('Выручка по ос')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Выручка по ос также схожа, худшими к концу года оказались macOS и Windows, которые упали сильнее всего во второй половине года.


In [ ]:
plt.figure(figsize=(8,6))
for os in arpau_month_os['os'].unique():
    df_plot = arpau_month_os[arpau_month_os['os'] == os].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['arpau'], marker='o', label=os)

plt.title('ARPAU по ос')
plt.xlabel('Месяц')
plt.ylabel('ARPAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


ARPAU также схожи, можно также обнаружить более быстрое падение macOS и Windows. При этом macOS долгое время лидировал. Единственной ос, которая вырасла на последнем месяце стала Android


In [ ]:
plt.figure(figsize=(8,6))
for os in conversion_month_os['os'].unique():
    df_plot = conversion_month_os[conversion_month_os['os'] == os].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['cr'], marker='o', label=os)

plt.title('Динамика cr по ос')
plt.xlabel('Дата')
plt.ylabel('cr')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Конверсия падает во всех сегментах начиная с мая, значения достают 40% в пике, падая до 20% в конце года


#### Сегментация по `age`


In [ ]:
# Добавим к events и orders информацию о возрасте покупателя
# Добавляем возраст через map
age_map = users.set_index('user_id')['age']
events['age'] = events['user_id'].map(age_map)
orders['age'] = orders['user_id'].map(age_map)

# Определяем границы и метки
bins = [0, 25, 35, 45, 60, 100]
labels = ['18-25', '26-35', '36-45', '46-60', '60+']

# Создаём группы
events['age_group'] = pd.cut(events['age'], bins=bins, labels=labels, right=False)
orders['age_group'] = pd.cut(orders['age'], bins=bins, labels=labels, right=False)

# Посчитаем количество активных пользователей по месяцам с учетом возраста
active_users_month_age = (events.groupby(['event_month', 'age_group'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))
# Выручка по месяцам и возрасту
revenue_month_age = (orders.groupby(['order_month', 'age_group'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))
revenue_month_age['revenue'] = revenue_month_age['total_sum'] * 0.05

# ARPAU
arpau_month_age = active_users_month_age.merge(
    revenue_month_age,
    how='left',
    left_on=['event_month', 'age_group'],
    right_on=['order_month', 'age_group']
)
# Расчитываем ARPAU
arpau_month_age['arpau'] = (arpau_month_age['revenue'] / arpau_month_age['active_users']).fillna(0)

# MPU
mpu_age = orders.groupby(['order_month', 'age_group'])['user_id'].nunique().reset_index(name='mpu')

# Конверсия
conversion_month_age = active_users_month_age.merge(
    mpu_age,
    how='left',
    left_on=['event_month', 'age_group'],
    right_on=['order_month', 'age_group']
)
conversion_month_age['cr'] = (conversion_month_age['mpu'] / 
                                 conversion_month_age['active_users'] * 100).fillna(0).round(2)

plt.figure(figsize=(8,6))
for age_group in arpau_month_age['age_group'].unique():
    df_plot = arpau_month_age[arpau_month_age['age_group'] == age_group].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['active_users'], marker='o', label=age_group)

plt.title('MAU по возрасту')
plt.xlabel('Месяц')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Обнаруживаем интересную динамику: большая часть пользователей приложения в возрасте от 45 лет. Падение метрики у этих возрастных групп не столь удручающее, а у 60+ и вовсе в последние два месяца происходит рост. Главной проблемой стоит считать недостаточный рост у более молодых групп: до 25 и 26-35, что достаточно интересно, учитывая огромный скачок роста канала привлечения TikTok во второй половине года.


In [ ]:
plt.figure(figsize=(8,6))
for age_group in arpau_month_age['age_group'].unique():
    df_plot = arpau_month_age[arpau_month_age['age_group'] == age_group].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['revenue'], marker='o', label=age_group)

plt.title('Выручка по возрасту')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Две самые платежеспособные группы: 46-60 и 26-35, однако они падают после августа. Низкие показатели 18-25 объяснимы - у молодых людей, вероятно, еще нет такой финансовой независимости и возможности постоянно совершать покупки как у людей постарше.


In [ ]:
plt.figure(figsize=(8,6))
for age_group in arpau_month_age['age_group'].unique():
    df_plot = arpau_month_age[arpau_month_age['age_group'] == age_group].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['arpau'], marker='o', label=age_group)

plt.title('ARPAU по возрасту')
plt.xlabel('Месяц')
plt.ylabel('ARPAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика arpau показывает преимущество у группы 26-35 на протяжении большей части года, в октябре их перегоняет группа 18-25, однако в конце года 26-35 вновь выходит вперед. Получается, что именно возрастные группы 26-35 и 46-60 являются для нас самыми важными, первая - показывает лучшую выручку на активного пользователя, вторая - самое большое число активных пользователей и выручку - именна эта группа имеют наибольшую перспективу для использования таргетированной рекламы в приложении и повышению их активности.


In [ ]:
plt.figure(figsize=(8,6))
for age_group in conversion_month_age['age_group'].unique():
    df_plot = conversion_month_age[conversion_month_age['age_group'] == age_group].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['cr'], marker='o', label=age_group)

plt.title('Динамика cr по возрасту')
plt.xlabel('Дата')
plt.ylabel('cr')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика конверсии схожа во всех возрастных группах. Самый высокий результат показала группа 18-25, у которой в первый месяц 50%. Но все группы приходят к менее чем 25% в декабре. Лучшими среди них является группа 26-35


#### Сегментация по `gender`


In [ ]:
# Добавим к events и orders информацию о поле покупателя
# Добавляем возраст через map
gender_map = users.set_index('user_id')['gender']
events['gender'] = events['user_id'].map(gender_map)
orders['gender'] = orders['user_id'].map(gender_map)

# Посчитаем количество активных пользователей по месяцам с учетом пола
active_users_month_gender = (events.groupby(['event_month', 'gender'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))

# Выручка по месяцам и полу
revenue_month_gender = (orders.groupby(['order_month', 'gender'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))
revenue_month_gender['revenue'] = revenue_month_gender['total_sum'] * 0.05

# ARPAU
arpau_month_gender = active_users_month_gender.merge(
    revenue_month_gender,
    how='left',
    left_on=['event_month', 'gender'],
    right_on=['order_month', 'gender']
)
# Расчитываем ARPAU
arpau_month_gender['arpau'] = (arpau_month_gender['revenue'] / arpau_month_gender['active_users']).fillna(0)

# MPU
mpu_gender = orders.groupby(['order_month', 'gender'])['user_id'].nunique().reset_index(name='mpu')

# Конверсия
conversion_month_gender = active_users_month_gender.merge(
    mpu_gender,
    how='left',
    left_on=['event_month', 'gender'],
    right_on=['order_month', 'gender']
)
conversion_month_gender['cr'] = (conversion_month_gender['mpu'] / 
                                 conversion_month_gender['active_users'] * 100).fillna(0).round(2)

plt.figure(figsize=(8,6))
for gender in arpau_month_gender['gender'].unique():
    df_plot = arpau_month_gender[arpau_month_gender['gender'] == gender].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['active_users'], marker='o', label=gender)

plt.title('MAU по полу')
plt.xlabel('Месяц')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Активность у женщин и мужчин практически одинаковая на протяжении всего периода


In [ ]:
plt.figure(figsize=(8,6))
for gender in arpau_month_gender['gender'].unique():
    df_plot = arpau_month_gender[arpau_month_gender['gender'] == gender].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['revenue'], marker='o', label=gender)

plt.title('Выручка по полу')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.axhline(arpau_month_gender[arpau_month_gender['gender']=='M']['revenue'].mean(), linestyle='--', color='purple', label='M_avg')
plt.axhline(arpau_month_gender[arpau_month_gender['gender']=='F']['revenue'].mean(), linestyle='--', color='steelblue', label='F_avg')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика выручки по полу также незначительно отличается, средняя выручка у женщин за период лишь немного выше благодаря более медленному падению в начале осени


In [ ]:
plt.figure(figsize=(8,6))
for gender in arpau_month_gender['gender'].unique():
    df_plot = arpau_month_gender[arpau_month_gender['gender'] == gender].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['arpau'], marker='o', label=gender)

plt.title('ARPAU по полу')
plt.xlabel('Месяц')
plt.ylabel('ARPAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Динамика ARPAU повторяет динамику выручки, к конце года женщины также вырываются вперед


In [ ]:
plt.figure(figsize=(8,6))
for gender in conversion_month_gender['gender'].unique():
    df_plot = conversion_month_gender[conversion_month_gender['gender'] == gender].sort_values('event_month')
    plt.plot(df_plot['event_month'], df_plot['cr'], marker='o', label=gender)

plt.title('Динамика cr по полу')
plt.xlabel('Дата')
plt.ylabel('cr')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Конверсия у мужчин держится хуже - стабильное падение начинается уже в мае и не прекращается до конца года. В то же время у женщин есть успешный июль и небольшой отскок в декабре


#### Сегментация по `region`


In [ ]:
# Добавим к events и orders информацию о поле покупателя
# Добавляем возраст через map
region_map = users.set_index('user_id')['region']
events['region'] = events['user_id'].map(region_map).fillna(0)
orders['region'] = orders['user_id'].map(region_map).fillna(0)

# Посчитаем количество активных пользователей по месяцам с учетом пола
active_users_month_region = (events.groupby(['event_month', 'region'])['user_id']
                               .nunique()
                               .reset_index(name='active_users'))

# Выручка по месяцам и полу
revenue_month_region = (orders.groupby(['order_month', 'region'])['total_price']
                          .sum()
                          .reset_index(name='total_sum'))
revenue_month_region['revenue'] = revenue_month_region['total_sum'] * 0.05

# ARPAU
arpau_month_region = active_users_month_region.merge(
    revenue_month_region,
    how='left',
    left_on=['event_month', 'region'],
    right_on=['order_month', 'region']
)
# Расчитываем ARPAU
arpau_month_region['arpau'] = (arpau_month_region['revenue'] / arpau_month_region['active_users']).fillna(0)

# MPU
mpu_region = orders.groupby(['order_month', 'region'])['user_id'].nunique().reset_index(name='mpu')

# Конверсия
conversion_month_region = active_users_month_region.merge(
    mpu_region,
    how='left',
    left_on=['event_month', 'region'],
    right_on=['order_month', 'region']
)
conversion_month_region['cr'] = (conversion_month_region['mpu'] / 
                                 conversion_month_region['active_users'] * 100).fillna(0).round(2)


plt.figure(figsize=(8,6))
for region in arpau_month_region['region'].unique():
    df_plot = arpau_month_region[arpau_month_region['region'] == region].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['active_users'], marker='o', label=region)

plt.title('MAU по региону')
plt.xlabel('Месяц')
plt.ylabel('MAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Наиболее активно посещают приложение пользователи из провинций - категории Другие регионы. Затем следуют пользователи из Москвы, Санкт-Петербурга и Московской области. Показатели других городов-миллионников прмерно равны


In [ ]:
plt.figure(figsize=(8,6))
for region in arpau_month_region['region'].unique():
    df_plot = arpau_month_region[arpau_month_region['region'] == region].sort_values('order_month')
    plt.plot(df_plot['event_month'], df_plot['revenue'], marker='o', label=region)

plt.title('Выручка по региону')
plt.xlabel('Месяц')
plt.ylabel('Выручка')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


В исследуемых нами регионах лидируют по выручке также Другие регионы, за ними следуют Москва, Санкт-Петербург и четвертой идет Московская область


<div class="alert alert-info"> <b>Комментарий студента:</b> 
Странно, но тут, ничего не меняя, выручка прогрузилась


In [ ]:
</div>
    

plt.figure(figsize=(12,6))
for region in arpau_month_region['region'].unique():
    df_plot = arpau_month_region[arpau_month_region['region'] == region].sort_values('order_month')
    plt.plot(df_plot['event_month'], df_plot['arpau'], marker='o', label=region)

plt.title('ARPAU по региону')
plt.xlabel('Месяц')
plt.ylabel('ARPAU')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


График ARPAU показывает, что у лидеров метрика имеет более стабильную динамику - очень схожую на общее ARPAU. В то же время города поменьше показывают высокую волатильность, например Ростов и Екатеринбург от месяца к месяце могут внезапно выходить на первое место, но на следующий падать в самый низ


In [ ]:
plt.figure(figsize=(12,6))
for region in conversion_month_region['region'].unique():
    df_plot = conversion_month_region[conversion_month_region['region'] == region].sort_values('order_month')
    plt.plot(df_plot['order_month'], df_plot['cr'], marker='o', label=region)

plt.title('Динамика cr по регионам')
plt.xlabel('Дата')
plt.ylabel('cr')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Конверсия в регионах очень схожая, выделяется Екатеринбург, который достигает более чем 50% в мае и июле, но падает на предпоследнее место к концу года. Москва и Санкт-Петербург не выделяются, как и Другие регионы


### Воронки


In [ ]:
# Определимся, какие события следует включать в воронку
events['event_type'].unique().tolist()

events.groupby('event_type')['user_id'].nunique().sort_values()


Нам нужны события, которые напрямую показывают путь пользователя от выбора товара до его приобретения, тогда основная ворона булет выглядеть так:


In [ ]:
- `page_view`
- `product_view`
- `product_click`
- `add_to_cart`
- `checkout_start`
- `checkout_complete`


Также расчитаем отдельные воронки для пользователей которые зарегистрировались, по разным каналам регистрации, покупательскому сегменту, девайсам и ос.


#### Общая воронка


In [ ]:
# Выберем необходимые нам события
order = ['page_view', 'product_view', 'product_click', 'add_to_cart', 'checkout_start', 'checkout_complete']
funnel_df = events[events['event_type'].isin(order)]

# Группировка и приведение к категориям
funnel_grouped = funnel_df.groupby('event_type')['user_id'].nunique().reset_index(name='users')
funnel_grouped['event_type'] = pd.Categorical(funnel_grouped['event_type'], categories=order, ordered=True)
funnel_grouped = funnel_grouped.sort_values('event_type').reset_index(drop=True)

# Расчёт конверсий
funnel_grouped['cr'] = (funnel_grouped['users'] / funnel_grouped['users'].max()) * 100
funnel_grouped['cr_step_by_step'] = (funnel_grouped['users'] / funnel_grouped['users'].shift(1)) * 100

# Для первого шага поставим 100 (или NaN)
funnel_grouped.loc[0, 'cr_step_by_step'] = 100.0


funnel_grouped= funnel_grouped.sort_values('cr')

plt.figure(figsize=(8,6))
plt.barh(funnel_grouped['event_type'], funnel_grouped['cr'], color='steelblue')
# Создадим функцию для вывода долей в процентах
for index,value in enumerate(funnel_grouped['cr']):
    plt.text(
        x = value,
        y = index,
        s = f'{value:.2f}')
plt.title('Воронка конверсии из просмотра в покупку')
plt.xlabel('Конверсия')
plt.ylabel('Событие')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
plt.barh(funnel_grouped['event_type'], funnel_grouped['cr_step_by_step'], color='steelblue')
# Создадим функцию для вывода долей в процентах
for index,value in enumerate(funnel_grouped['cr_step_by_step']):
    plt.text(
        x = value,
        y = index,
        s = f'{value:.2f}')
plt.title('Воронка конверсии из просмотра в покупку')
plt.xlabel('Конверсия')
plt.ylabel('Событие')
plt.tight_layout()
plt.show()


Обе воронки показывают хороший результат вплоть до начала оплаты: в простой воронке теряется 16 процентов, еще 13% не заканчивают оплату товара. Step-by-step подтверждает, что главный спад происходит на этапе начала оплаты - только 82%. Столько же из начавших оплату ее заканчивают.


#### Воронка залогиненных пользователей


In [ ]:
# Отфильтруем датафрейм, оставим только залогиненых пользователей
logged_users = events[events['event_type']=='user_login']['user_id']
df_funnel_logged = events[events['user_id'].isin(logged_users)]

funnel_grouped_logged = df_funnel_logged.groupby('event_type')['user_id'].nunique().reset_index(name='users')
funnel_grouped_logged['event_type'] = pd.Categorical(funnel_grouped_logged['event_type'], categories=order, ordered=True)
funnel_grouped_logged = funnel_grouped_logged.sort_values('event_type').reset_index(drop=True)

# Расчёт конверсий
funnel_grouped_logged['cr'] = (funnel_grouped_logged['users'] / funnel_grouped_logged['users'].max()) * 100
funnel_grouped_logged['cr_step_by_step'] = (funnel_grouped_logged['users'] / funnel_grouped_logged['users'].shift(1)) * 100
funnel_grouped_logged.loc[0, 'cr_step_by_step'] = 100.0

# Сортируем по логическому порядку
funnel_grouped_logged = funnel_grouped_logged.sort_values('event_type')

# Преобразуем event_type в строки и удаляем возможные пропуски
funnel_grouped_logged = funnel_grouped_logged.dropna(subset=['event_type'])
funnel_grouped_logged['event_type'] = funnel_grouped_logged['event_type'].astype(str)
# Если остались строки 'nan', удалим их
funnel_grouped_logged = funnel_grouped_logged[funnel_grouped_logged['event_type'] != 'nan']

funnel_grouped_logged= funnel_grouped_logged.sort_values('cr')

plt.figure(figsize=(8,6))
plt.barh(funnel_grouped_logged['event_type'], funnel_grouped_logged['cr'], color='steelblue')
# Создадим функцию для вывода долей в процентах
for index,value in enumerate(funnel_grouped_logged['cr']):
    plt.text(
        x = value,
        y = index,
        s = f'{value:.2f}')
plt.title('Воронка конверсии из просмотра в покупку')
plt.xlabel('Конверсия')
plt.ylabel('Событие')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,6))
plt.barh(funnel_grouped_logged['event_type'], funnel_grouped_logged['cr_step_by_step'], color='steelblue')
# Создадим функцию для вывода долей в процентах
for index,value in enumerate(funnel_grouped_logged['cr_step_by_step']):
    plt.text(
        x = value,
        y = index,
        s = f'{value:.2f}')
plt.title('Воронка конверсии из просмотра в покупку')
plt.xlabel('Конверсия')
plt.ylabel('Событие')
plt.tight_layout()
plt.show()


Среди зарегистрированных пользователей воронка замтено лучше: даже на шаге добавления в корзину конверсия 97%, начинают покупку уже 89%, а заканчивают 79%. Step-by-step воронка также показывает отсутсвие каких-либо провалов в пути пользователя. Почти 80% - очень хороший показатель конверсии.


#### Воронка по каналу привлечения


In [ ]:
# Сгруппируем по событию и каналу
funnel_channel = funnel_df.groupby(['event_type', 'acq_channel'])['user_id'].nunique().reset_index(name='users')
funnel_channel['event_type'] = pd.Categorical(funnel_channel['event_type'], categories=order, ordered=True)
funnel_channel = funnel_channel.sort_values(['acq_channel', 'event_type']).reset_index(drop=True)

channels = funnel_channel['acq_channel'].unique()

# Позиции этапов на оси X
x = np.arange(len(order))
# Ширина столбца зависит от количества каналов
width = 0.8 / len(channels)

fig, ax = plt.subplots(figsize=(12, 7))

for i, channel in enumerate(channels):
    df_bar = (
        funnel_channel[funnel_channel['acq_channel'] == channel]
        .set_index('event_type')
        .reindex(order)
        .reset_index())
    # Конверсия относительно первого этапа
    first_step_users = df_bar.loc[0, 'users']

    df_bar['cr'] = (df_bar['users'] / first_step_users * 100).round(2)
    # Смещение столбцов для каждого канала
    positions = x - 0.4 + width / 2 + i * width
    bars = ax.bar(
        positions,
        df_bar['cr'],
        width=width,
        label=channel
    )
ax.set_title('Воронка конверсии из просмотра в покупку')
ax.set_xlabel('Событие')
ax.set_ylabel('Конверсия от первого этапа')

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


Воронка по каналам привлечения показывает нам, что TikTok, несмотря на то, что является самым быстрорастущим каналом привлечения, не дает нам достаточно платящих пользователей - юзеры отсюда только в 40% случаев доходят до покупки при просмотре товала. Лучше всего себя проявляют каналы SEO и Affiliate - оба в тройке самых дешевых.


#### Воронка по покупательскому сегменту


In [ ]:
# Сгруппируем по событию и сегменту
funnel_segment = funnel_df.groupby(['event_type', 'buyer_segment'])['user_id'].nunique().reset_index(name='users')
funnel_segment['event_type'] = pd.Categorical(funnel_segment['event_type'], categories=order, ordered=True)
funnel_segment = funnel_segment.sort_values(['buyer_segment', 'event_type']).reset_index(drop=True)

segments = funnel_segment['buyer_segment'].unique()

# Позиции этапов на оси X
x = np.arange(len(order))
# Ширина столбца зависит от количества сегментов
width = 0.8 / len(segments)

fig, ax = plt.subplots(figsize=(12, 7))

for i, segment in enumerate(segments):
    df_bar = (
        funnel_segment[funnel_segment['buyer_segment'] == segment]
        .set_index('event_type')
        .reindex(order)
        .reset_index())
    # Конверсия относительно первого этапа
    first_step_users = df_bar.loc[0, 'users']

    df_bar['cr'] = (df_bar['users'] / first_step_users * 100).round(2)
    # Смещение столбцов для каждого канала
    positions = x - 0.4 + width / 2 + i * width
    bars = ax.bar(
        positions,
        df_bar['cr'],
        width=width,
        label=segment
    )
ax.set_title('Воронка конверсии из просмотра в покупку')
ax.set_xlabel('Событие')
ax.set_ylabel('Конверсия от первого этапа')

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


Лучше всего конверсия у regular пользователей - именно они приносят больше всего выручки, более 80% из них завершают покупки. Хуже всего конверсия у one_time пользователей. Сюда относятся юзеры, которые не совершили более одной покупки - им, скорее всего, не понравился интерфейс или же они передумали покупать товар


#### Воронка по девайсам


In [ ]:
# Сгруппируем по событию и девайсу
funnel_device = funnel_df.groupby(['event_type', 'device'])['user_id'].nunique().reset_index(name='users')
funnel_device['event_type'] = pd.Categorical(funnel_device['event_type'], categories=order, ordered=True)
funnel_device = funnel_device.sort_values(['device', 'event_type']).reset_index(drop=True)

devices = funnel_device['device'].unique()

# Позиции этапов на оси X
x = np.arange(len(order))
# Ширина столбца зависит от количества сегментов
width = 0.8 / len(devices)

fig, ax = plt.subplots(figsize=(12, 7))

for i, device in enumerate(devices):
    df_bar = (
        funnel_device[funnel_device['device'] == device]
        .set_index('event_type')
        .reindex(order)
        .reset_index())
    # Конверсия относительно первого этапа
    first_step_users = df_bar.loc[0, 'users']

    df_bar['cr'] = (df_bar['users'] / first_step_users * 100).round(2)
    # Смещение столбцов для каждого канала
    positions = x - 0.4 + width / 2 + i * width
    bars = ax.bar(
        positions,
        df_bar['cr'],
        width=width,
        label=device
    )
ax.set_title('Воронка конверсии из просмотра в покупку')
ax.set_xlabel('Событие')
ax.set_ylabel('Конверсия от первого этапа')

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


Воронка показывает, что конверсия на всех устройствах очень схожа - несколько лучше остальных только десктопные юзеры, но они обгоняют остальных на 1-2%.


#### Воронка по ос


In [ ]:
# Сгруппируем по событию и ос
funnel_os = funnel_df.groupby(['event_type', 'os'])['user_id'].nunique().reset_index(name='users')
funnel_os['event_type'] = pd.Categorical(funnel_os['event_type'], categories=order, ordered=True)
funnel_os = funnel_os.sort_values(['os', 'event_type']).reset_index(drop=True)

oss = funnel_os['os'].unique()

# Позиции этапов на оси X
x = np.arange(len(order))
# Ширина столбца зависит от количества сегментов
width = 0.8 / len(oss)

fig, ax = plt.subplots(figsize=(12, 7))

for i, os in enumerate(oss):
    df_bar = (
        funnel_os[funnel_os['os'] == os]
        .set_index('event_type')
        .reindex(order)
        .reset_index())
    # Конверсия относительно первого этапа
    first_step_users = df_bar.loc[0, 'users']

    df_bar['cr'] = (df_bar['users'] / first_step_users * 100).round(2)
    # Смещение столбцов для каждого канала
    positions = x - 0.4 + width / 2 + i * width
    bars = ax.bar(
        positions,
        df_bar['cr'],
        width=width,
        label=os
    )
ax.set_title('Воронка конверсии из просмотра в покупку')
ax.set_xlabel('Событие')
ax.set_ylabel('Конверсия от первого этапа')

ax.set_xticks(x)
ax.set_xticklabels(order, rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


Обнаруживаем, что по ос также нет значительных различий в конверсии - лучше остальных на 1% проявляет себя android.


## Выводы по этапу 4

    
Ключевые проблемы продукта - низкая конверсия у самого большого канала - TikTok, и проблемы в воронке у пользователей с одним заказом, также проблемой является падение активности пользователей, привлеченных с помощью Google ads - они стоят дороже всех, но не окупают вложений.
 
Гипотеза: каналы с высоким CAC  привлекают пользователей с низкой конверсией и LTV.
Действие: перераспределить бюджет в пользу SEO, других дешевых каналов.
    
Гипотеза: Пользователи из TikTok имеют проблемы с конверсией в заказ
Действие: Предложить пользователям из TikTok бонус на первый заказ, например скидку.
    
Гипотеза: Конверсия и выручка не растут в ключевые периоды: новый год, черная пятница и т.д.
Действие: Добавить дополнительные бонусы, скидочные программы специально для ключевых периодов, протестировать уже в рождество.


## 5. Подготовка эксперимента и подведение его результатов


# Новые вводные

Команда продукта рассмотрела твои гипотезы и на основании этого решила влиять на конверсию при помощи финансовых стимуляций. Так как ситуация со снижением выручки срочная и безотлагательная, пока ты продолжал изучать данные, они подключили другого аналитика для подготовки дизайн документа, но само проведение эксперимента на тебе. При формировании выводов допиши рекомендации по итогам теста и, если необходимо, опиши проблемы, присутствующие в эксперименте.


### Новые данные:

```python
pa_diploma_events_AB = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_events_AB.csv')
pa_diploma_orders_AB = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_orders_AB.csv')
pa_diploma_sessions_AB = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_sessions_AB.csv')
pa_diploma_users_AB = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_users_AB.csv')
```


### Результаты сплита лежат в таблице:

```python
pa_diploma_AB_split_users = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_AB_split_users.csv')
```

* user_id – id клиента
* group – группа, в которую попал клиент

Таблицы с данными о пользователях, событиях и заказах идентичны таблицам с историческими данными.

Описание таблицы `sessions_AB`

| Поле | Описание |
|------|----------|
| `session_id` | Уникальный идентификатор сессии. |
| `user_id` | Уникальный идентификатор пользователя. |
| `session_start` | Дата и время начала сессии. |
| `os` | Операционная система устройства. |
| `device` | Тип устройства. |
| `region` | Регион пользователя. |
| `browser` | Браузер, используемый пользователем. |
| `country` | Страна пользователя. |
| `entry_path` | Путь входа (начальная страница сессии). |
| `path_start` | Путь начала отслеживания. |
| `utm_source` |  |
| `screen_size` | Разрешение экрана устройства. |
| `scroll_depth` | Глубина прокрутки. |
| `user_segment` | Сегмент пользователя (regular/premium и т.д.). |
| `utm_campaign_id` | Идентификатор рекламной кампании. |
| `session_week` | Неделя сессии. |
| `session_month` | Месяц сессии. |


## Дизайн эксперимента:

Бонус на первую покупку для пользователей TikTok


### 1. Контекст и цель эксперимента

- **Бизнес-контекст:** Команда продукта запускает эксперимент, направленный на повышение конверсии среди пользователей, привлечённых через канал TikTok. Для стимулирования первой покупки пользователям будет предоставлен бонус на первую покупку.  
- **Цель:** Проверить, влияет ли предоставление бонуса на первую покупку на конверсию и выручку среди пользователей TikTok.  
- **Гипотеза:** Предоставление бонуса на первую покупку новым пользователям, привлеченным через TikTok, приведет к увеличению конверсии в первую покупку

---


### 2. Формат эксперимента

- **Тип эксперимента:** Сплит-роллинг (rolling split)  
- **Канал:** Только пользователи, пришедшие с TikTok  
- **Группы:**  
  - **Контрольная (A):** пользователи TikTok без бонуса  
  - **Тестовая (B):** пользователи TikTok с бонусом на первую покупку  
- **Распределение:** Равномерное распределение новых пользователей TikTok между группами по дате регистрации  
- **Период проведения:** 3 недели (фиксировано)

---


### 3. Ключевые метрики


#### Основная метрика

- **Конверсия в первую покупку** — доля пользователей, зарегистрировавшихся с TikTok и совершивших первую покупку  
  - *Цель:* увеличить конверсию с помощью бонуса  
  - *Критерий успеха:* статистически значимый рост конверсии в тестовой группе


#### Вспомогательные метрики

- **ARPU** (средняя выручка на пользователя TikTok)  
- **ARPPU** (средняя выручка на покупателя TikTok)  
- **AOV** (средний чек TikTok)


#### Барьерная метрика

- **Общая выручка от пользователей TikTok**

---


### 4. Размеры групп и запуск

- **Целевая аудитория:** только пользователи TikTok  
- **Размеры групп:** определяются по фактическому трафику TikTok за период эксперимента  
- **Комментарий:** запуск без предварительного расчёта мощности и MDE из-за срочности задачи


### Посчитаем метрики по историчеким данным


In [ ]:
# Пользователи TikTok
tiktok_users = users.loc[users['acq_channel'] == 'TikTok', ['user_id', 'acq_channel', 'cohort_month']].copy()

# Определяем, совершили ли они хотя бы один заказ
first_orders = orders.groupby('user_id')['order_date'].min().reset_index(name='first_order_date')
tiktok_users = tiktok_users.merge(first_orders, on='user_id', how='left')
tiktok_users['has_purchased'] = tiktok_users['first_order_date'].notna()

# Посчитаем основную метрику
df_hist = tiktok_users.groupby('cohort_month').agg(
    users=('user_id','nunique'),
    buyers=('has_purchased','sum')
).reset_index()

df_hist['kpi'] = (df_hist['buyers']/df_hist['users']*100).round(2)

# Посчитаем барьерную метрику
tiktok_orders = orders.loc[orders['acq_channel'] == 'TikTok', ['user_id', 'order_month', 'total_price']]
tiktok_revenue = tiktok_orders.groupby('order_month')['total_price'].sum().reset_index(name='total_sum')
# Переименовываем столбец для объединения
tiktok_revenue = tiktok_revenue.rename(columns={'order_month': 'cohort_month'})
tiktok_revenue['revenue'] = (tiktok_revenue['total_sum'] * 0.05).round(2)

# Присоединяем к df_hist
df_hist = df_hist.merge(tiktok_revenue[['cohort_month', 'revenue']], how='left', on='cohort_month')

# Посчитаем вспомогательные метрики
df_hist['arpu'] = (df_hist['revenue']/df_hist['users']).round(2)
df_hist['arppu'] = (df_hist['revenue']/df_hist['buyers']).round(2)

aov_month_tiktok = tiktok_orders.groupby('order_month')['total_price'].mean().reset_index(name='aov')
aov_month_tiktok = aov_month_tiktok.rename(columns={'order_month':'cohort_month'})
df_hist = df_hist.merge(aov_month_tiktok[['cohort_month', 'aov']], how='left', on='cohort_month')
df_hist['aov'] = (df_hist['aov']).round(2)

def plot_func(x, y, 
              marker='o',
              alpha = 0.3, 
              title='',
              xlabel = 'Месяц',
              ylabel=''):
    plt.plot(x, y, marker=marker)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(alpha=alpha)
    plt.tight_layout()
    plt.show

plt.figure(figsize=(8,6))
plot_func(df_hist['cohort_month'],
         df_hist['kpi'],
         title = 'Динамика ключевой метрики на исторических данных',
         ylabel = 'KPI')


Ключевая метрика статична до июня, совершает скачок в июле, но затем падает до конца года, как раз со скачком роста числа пользователей из TikTok


In [ ]:
plt.figure(figsize=(8,6))
plot_func(df_hist['cohort_month'],
         df_hist['revenue'],
         title = 'Динамика берьерной метрики на исторических данных',
         ylabel = 'Выручка')


Выручка, однако растет в больших объемах, барьерная метрика не проседает


In [ ]:
plt.figure(figsize=(8,6))
plot_func(df_hist['cohort_month'],
         df_hist['arpu'],
         title = 'Динамика arpu на исторических данных',
         ylabel = 'arpu')


ARPU показывает значительные скачки на протяжении года - меняясь от 25 до 80 от одного месяца к другому. Но во второй половине года метрика стабилизируется на уровне 40-50


In [ ]:
plt.figure(figsize=(8,6))
plot_func(df_hist['cohort_month'],
         df_hist['arppu'],
         title = 'Динамика arppu на исторических данных',
         ylabel = 'arppu')


ARPPU также волатильна, она скачет от 500 до 1700 в первой половине года. В декабре совершает скачок до почти 3000


In [ ]:
plt.figure(figsize=(8,6))
plot_func(df_hist['cohort_month'],
         df_hist['aov'],
         title = 'Динамика aov на исторических данных',
         ylabel = 'aov')


Динамика среднего чека похожа на динамику arpu, однако он начинает год на пике. Самый большой провал в Марте. Этот месяц вовсе стал худшим по всем метрикам для пользователей из TikTok.


### Предобработка данных


#### Таблица `split_users`


In [ ]:
split_users = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_AB_split_users.csv')

split_users

split_users.info()

split_users['user_id'].duplicated().sum()

split_users['group'].unique()

user_group_count = split_users.groupby('user_id')['group'].nunique()
users_in_multiple = user_group_count[user_group_count > 1].index.tolist()
users_in_multiple


В данных о сплите не оказалось пропусков и дубликатов, ни один пользователь не попал в две группы, проверим, как распределились пользователи


In [ ]:
split_results = split_users.groupby('group')['user_id'].nunique().reset_index(name='group_users')
split_results['share'] = (split_results['group_users'] / split_results['group_users'].sum()).round(2)

split_results


Пользователи распределились по группам равномерно - почти 50 на 50


#### Таблица `events`


In [ ]:
# Загрузим новые данные
events_test = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_events_AB.csv', parse_dates=[3,-2,-1])

# Оставим только пользователей, учавствовавших в сплитовании
events_test = events_test[events_test['user_id'].isin(split_users['user_id'])]

events_test.head()

events_test.info()

events_test['event_date'].dt.normalize().unique()


В таблице содержатся данные, не входящие в нужный нам период. Нам нужно оставить только первые известные нам три недели 2025 года.


In [ ]:
events_test = events_test[(events_test['event_date']>='2024-12-31') &
                         (events_test['event_date']<'2025-01-22')]


#### Таблица `sessions`


In [ ]:
sessions_test = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_sessions_AB.csv', parse_dates=[2,-2,-1])

sessions_test = sessions_test[sessions_test['user_id'].isin(split_users['user_id'])]

sessions_test.head()

sessions_test.info()

sessions_test = sessions_test[(sessions_test['session_start']>='2024-12-31') &
                         (sessions_test['session_start']<'2025-01-22')]


#### Таблица `orders`


In [ ]:
orders_test = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_orders_AB.csv', parse_dates=[2,-2,-1])

orders_test = orders_test[orders_test['user_id'].isin(split_users['user_id'])]

orders_test.head(10)

orders_test.info()

orders_test = orders_test[(orders_test['order_date']>='2024-12-31') &
                         (orders_test['order_date']<'2025-01-22')]


#### Таблица `users`


In [ ]:
users_test = pd.read_csv('https://code.s3.yandex.net/datasets/pa_diploma_users_AB.csv', parse_dates=[1,-2,-1])

users_test = users_test[users_test['user_id'].isin(split_users['user_id'])]
users_test = users_test[users_test['acq_channel']=='TikTok']

users_test.head()

users_test.info()

users_test = users_test[(users_test['registration_date']>='2024-12-31') &
                         (users_test['registration_date']<'2025-01-22')]


### Расчет метрик


In [ ]:
# Добавим в таблицы информацию о группах
groups_map = split_users.set_index('user_id')['group']

events_test['group'] = events_test['user_id'].map(groups_map)
sessions_test['group'] = sessions_test['user_id'].map(groups_map)
orders_test['group'] = orders_test['user_id'].map(groups_map)
users_test['group'] = users_test['user_id'].map(groups_map)


#### KPI

Наша метрика - Долевая, будем использовать z-тест пропорций, тогда наши гипотезы таковы:

Н0: p0 = p1 - Конверсия у тестовой и контрольной группы остались равными

Н1: p0 < p1 - Конверсия у тестовой группы выросла


In [ ]:
# Посчитаем ключевую метрику
users_test['is_buyer'] = users_test['user_id'].isin(orders_test['user_id'])

users_test['reg_date'] = users_test['registration_date'].dt.floor('D')
df_test = users_test.groupby(['group','reg_date']).agg(
    users=('user_id', 'nunique'),
    buyers=('is_buyer', 'sum')
).reset_index()

df_test['kpi'] = (df_test['buyers']/df_test['users']*100).round(2)
df_test['rolling_kpi'] = df_test['kpi'].rolling(window=7, min_periods=1).mean()

plt.figure(figsize=(8,6))
for group in df_test['group'].unique():
    df_plot = df_test[df_test['group']==group].sort_values('reg_date')
    plt.plot(df_plot['reg_date'], df_plot['kpi'], marker='o', label=group)
    plt.plot(df_plot['reg_date'], df_plot['rolling_kpi'], label=f'{group} rolling', linestyle='--')
plt.title('Динамика ключевой метрики по группам')
plt.xlabel('Дата')
plt.ylabel('KPI')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


Динамика у контрольной и тестовой группы различается уже с первых дней - несмотря на большую волатильность, тестовая группа всегда лучше, контрольная же в некоторые дни имеет 0 в значении метрики. Отметим, что на третью неделю в тестовой группе произошел серьезный спад, но на выходных метрика начала выправляться


In [ ]:
conv = users_test.groupby('group').agg(
    users=('user_id', 'nunique'),
    buyers=('is_buyer', 'sum')
)
conv['conv_pct'] = (conv['buyers'] / conv['users'] * 100).round(2)
print(conv)

lift = (conv.loc['treatment', 'conv_pct'] / conv.loc['control', 'conv_pct'] - 1) * 100
print(f'Относительный прирост: {lift:.0f}%')

# Метрика долевая, значит мы должны использовать Z-тест
alpha = 0.05

# Агрегируем пользователей по группам
z_test = users_test.groupby(['group']).agg(
    users=('user_id', 'nunique'),
    buyers=('is_buyer', 'sum')
).reset_index()

# Извлекаем данные для групп
control = z_test[z_test['group'] == 'control']
test = z_test[z_test['group'] == 'treatment']

n_success = [control['buyers'].iloc[0], test['buyers'].iloc[0]]
n_total = [control['users'].iloc[0], test['users'].iloc[0]]


z_stat, p_value = proportions_ztest(n_success, n_total, alternative='smaller')

print(f'p-value: {p_value:.4f}')
if p_value<alpha:
    print('Рост метрики оказался статистически значимым, можем принять Н1')
else:
    print('Рост метрики не показал статистической значимости, недостаточно оснований отклонить Н1')


Рост ключевой метрики оказался статистически значимым, теперь остается првоерить дополнительные метрики чтобы признать эксперимент удачным


#### Revenue

Барьерная метрика - среднее, будем использовать t-тест, тогда наши гипотезы таковы:

Н0: μ0 = μ1 - Выручка у тестовой и контрольной группы остались равными

Н1: μ0 < μ1 - Выручка у тестовой группы выросла


In [ ]:
# Добавляем канал
channel_map = users_test.set_index('user_id')['acq_channel']
orders_test['acq_channel'] = orders_test['user_id'].map(channel_map)
orders_test = orders_test[orders_test['acq_channel'] == 'TikTok']

# Группируем по группе и дате
revenue_test = orders_test.groupby([
    orders_test['order_date'].dt.floor('D').rename('reg_date'),
    'group'
])['total_price'].sum().reset_index(name='total_sum')

# Считаем выручку и скользящее среднее
revenue_test['revenue'] = revenue_test['total_sum'] * 0.05
revenue_test['rolling_revenue'] = revenue_test['revenue'].rolling(window=7, min_periods=1).mean().fillna(0)

df_test = df_test.merge(revenue_test[['reg_date', 'group', 'revenue', 'rolling_revenue']],
                       how='left', on=['reg_date', 'group'])

df_test = df_test.fillna(0)

plt.figure(figsize=(8,6))
for group in df_test['group'].unique():
    df_plot = df_test[df_test['group']==group].sort_values('reg_date')
    plt.plot(df_plot['reg_date'], df_plot['revenue'], marker='o', label=group)
    plt.plot(df_plot['reg_date'], df_plot['rolling_revenue'], label=f'{group} rolling', linestyle='--')
plt.title('Динамика барьерной метрики по группам')
plt.xlabel('Дата')
plt.ylabel('Revenue')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


Динамика берьерной метрики у тестовой группы выглядит лучше, но она крайне волатильна - скачет от дня ко дню  на 50%


In [ ]:
revenue_ttest = orders_test.groupby(['user_id','group'])['total_price'].sum().reset_index(name='total_sum')

revenue_ttest['revenue'] = revenue_ttest['total_sum']*0.05

control_revenue = revenue_ttest[revenue_ttest['group'] == 'control']['revenue']
test_revenue = revenue_ttest[revenue_ttest['group'] == 'treatment']['revenue']

t_stat, p_value = ttest_ind(control_revenue, test_revenue, equal_var=False, alternative='less')

print(f'p-value: {p_value:.4f}')
if p_value<alpha:
    print('Рост метрики оказался статистически значимым, можем принять Н1')
else:
    print('Рост метрики не показал статистической значимости, недостаточно оснований отклонить Н1')


Рост барьерной метрики не оказался статистически значимым, однако для барьерной метрики это не столь значимо, главное - она не упала, но даже выросла.


#### ARPU

ARPU - среднее, будем использовать t-тест, тогда наши гипотезы таковы:

Н0: μ0 = μ1 - ARPU у тестовой и контрольной группы остались равными

Н1: μ0 < μ1 - ARPU у тестовой группы выросла


In [ ]:
df_test['arpu'] = df_test['revenue']/df_test['users']
df_test['rolling_arpu'] = df_test['arpu'].rolling(window=7, min_periods=1).mean()

plt.figure(figsize=(8,6))
for group in df_test['group'].unique():
    df_plot = df_test[df_test['group']==group].sort_values('reg_date')
    plt.plot(df_plot['reg_date'], df_plot['arpu'], marker='o', label=group)
    plt.plot(df_plot['reg_date'], df_plot['rolling_arpu'], label=f'{group} rolling', linestyle='--')
plt.title('Динамика arpu по группам')
plt.xlabel('Дата')
plt.ylabel('arpu')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


ARPU в тестовой группе показывает стремительный рост, начиная со второй недели, однако у нее и растет волатильность. В то же время контрольная группа практически не показывает роста


In [ ]:
all_users = users_test[['user_id', 'group']].drop_duplicates()

# Присоединяем выручку
revenue_ttest = all_users.merge(revenue_ttest[['user_id', 'group', 'revenue']],
                                on=['user_id', 'group'],
                                how='left').fillna(0)

# Определяем выборки для t-теста
control_revenue = revenue_ttest[revenue_ttest['group'] == 'control']['revenue']
test_revenue = revenue_ttest[revenue_ttest['group'] == 'treatment']['revenue']

t_stat, p_value = ttest_ind(control_revenue, test_revenue, equal_var=False, alternative='less')

print(f'p-value: {p_value:.4f}')
if p_value<alpha:
    print('Рост метрики оказался статистически значимым, можем принять Н1')
else:
    print('Рост метрики не показал статистической значимости, недостаточно оснований отклонить Н1')


ARPU показывает статистически значимый рост


#### ARPPU

ARPU - среднее, будем использовать t-тест, тогда наши гипотезы таковы:

Н0: μ0 = μ1 - ARPPU у тестовой и контрольной группы остались равными

Н1: μ0 < μ1 - ARPPU у тестовой группы выросла


In [ ]:
# Расчитываем arppu, избегая деление на 0
df_test['arppu'] = np.where(df_test['buyers'] > 0,
                           df_test['revenue'] / df_test['buyers'],
                           0)

df_test['rolling_arppu'] = df_test['arppu'].rolling(window=7, min_periods=1).mean()

plt.figure(figsize=(8,6))
for group in df_test['group'].unique():
    df_plot = df_test[df_test['group']==group].sort_values('reg_date')
    plt.plot(df_plot['reg_date'], df_plot['arppu'], marker='o', label=group)
    plt.plot(df_plot['reg_date'], df_plot['rolling_arppu'], label=f'{group} rolling', linestyle='--')
plt.title('Динамика arppu по группам')
plt.xlabel('Дата')
plt.ylabel('arppu')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


ARPPU также показывает лучшую динамику в тестовой группе, пока в контрольной метрика не растет


In [ ]:
# Группируем покупателей
buyers_revenue = orders_test.groupby(['user_id', 'group'])['total_price'].sum().reset_index()
buyers_revenue['revenue'] = buyers_revenue['total_price'] * 0.05

# Выборки по группам
control_rev = buyers_revenue[buyers_revenue['group'] == 'control']['revenue']
test_rev = buyers_revenue[buyers_revenue['group'] == 'treatment']['revenue']

t_stat, p_value = ttest_ind(control_rev, test_rev, equal_var=False, alternative='less')

print(f'p-value: {p_value:.4f}')
if p_value<alpha:
    print('Рост метрики оказался статистически значимым, можем принять Н1')
else:
    print('Рост метрики не показал статистической значимости, недостаточно оснований отклонить Н1')


Рост ARPPU не показал статистистической значимости


#### AOV

ARPU - среднее, будем использовать t-тест, тогда наши гипотезы таковы:

Н0: μ0 = μ1 - AOV у тестовой и контрольной группы остались равными

Н1: μ0 < μ1 - AOV у тестовой группы выросла


In [ ]:
# Группируем по группе и дате
revenue_test = orders_test.groupby([
    orders_test['order_date'].dt.floor('D').rename('reg_date'),
    'group'
])['total_price'].mean().reset_index(name='aov')

# Скользящее среднее за 7 дней 
revenue_test['aov_rolling'] = revenue_test.groupby('group')['aov'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# Объединяем с df_test
df_test = df_test.merge(revenue_test[['reg_date', 'group', 'aov', 'aov_rolling']],
                        how='left', on=['reg_date', 'group'])

df_test['aov'] = df_test['aov'].fillna(0)
df_test['aov_rolling'] = df_test['aov_rolling'].fillna(0)

# График
plt.figure(figsize=(8,6))
for group in df_test['group'].unique():
    df_plot = df_test[df_test['group'] == group].sort_values('reg_date')
    plt.plot(df_plot['reg_date'], df_plot['aov'], marker='o', label=f'{group}')
    plt.plot(df_plot['reg_date'], df_plot['aov_rolling'], linestyle='--', label=f'{group} rolling')
plt.title('Динамика AOV по группам')
plt.xlabel('Дата')
plt.ylabel('Средний чек (AOV)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


Метрика скачет у обеих групп, сложно выявить тенденцию


In [ ]:
# 1. Для каждого пользователя считаем средний чек (AOV)
user_aov = orders_test.groupby(['user_id', 'group']).agg(
    total_spent=('total_price', 'sum'),
    order_count=('order_id', 'count')
).reset_index()

# Избегаем деления на ноль
user_aov['aov'] = user_aov['total_spent'] / user_aov['order_count']

# Выборки по группам
control_aov = user_aov[user_aov['group'] == 'control']['aov']
test_aov = user_aov[user_aov['group'] == 'treatment']['aov']

t_stat, p_value = ttest_ind(control_aov, test_aov, equal_var=False, alternative='less')

print(f'p-value: {p_value:.4f}')
if p_value<alpha:
    print('Рост метрики оказался статистически значимым, можем принять Н1')
else:
    print('Рост метрики не показал статистической значимости, недостаточно оснований отклонить Н1')


Метрика не показала статистически значимого роста


## Выводы по этапу 5


### Исторический контекст:
До июня конверсия была стабильной, затем в июле резко выросла (что совпало с ростом трафика из TikTok), после чего до конца года падала. Это говорит о том, что качество трафика из TikTok нестабильно, и без стимулов конверсия возвращается к низким значениям.

---


### Результаты теста:

- Тестовая группа стабильно показывала более высокую конверсию, чем контрольная, даже с учётом высокой волатильности.

- На третьей неделе наблюдался спад в тестовой группе, но к выходным метрика начала восстанавливаться.

- p-value < 0.05 – рост конверсии статистически значим.

**Основная цель эксперимента достигнута – бонус эффективно увеличивает первую покупку у пользователей TikTok**. Рекомендуется внедрять бонус на постоянной основе, но с возможностью мониторинга долгосрочной динамики.


### Вспомогательные метрики


#### ARPU
Результаты теста:

- Тестовая группа показывает стремительный рост со второй недели, хотя волатильность также растёт.

- Контрольная группа практически не растёт.

- p-value < 0.05 – рост статистически значим.

**Бонус способствует росту выручки на пользователя – это позитивный сигнал**. ARPU растёт за счёт привлечения новых платящих клиентов.


#### ARPPU
Результаты теста:

- Тестовая группа показывает лучшую динамику, чем контрольная.

- Рост не статистически значим.

**Несмотря на отсутствие статистической значимости, тренд положительный**. Учитывая, что ARPPU исторически волатилен, отсутствие значимости не должно быть препятствием для внедрения, особенно если основная метрика выиграла.


#### AOV
Результаты теста:

- В обеих группах AOV скачет, устойчивой тенденции не выявлено.

- Рост не статистически значим.

**Бонус не влияет на средний чек, что логично – он стимулирует первую покупку, но не повышает сумму заказа**. Это нейтральный результат, не мешающий успеху эксперимента.


### Барьерная метрика
Результаты теста:

- В тестовой группе динамика лучше, но крайне волатильна (скачки до 50% день ото дня).

- Рост не статистически значим, но выручка не упала – она даже выросла.

**Для барьерной метрики главное – отсутствие падения**. Здесь выручка не просела, а показала положительную динамику, значит эксперимент безопасен с точки зрения общего дохода.

--- 


### Бизнес-рекоммендация

**Эксперимент успешен** – основная метрика и одна из вспомогательных показали статистически значимый рост. Барьерная метрика не ухудшилась, что позволяет рекомендовать внедрение бонуса для пользователей из TikTok на постоянной основе. Бизнесу можно внедрять *бонус для пользователей из TikTok на постоянной основе.*

**Следующие возможные шаги**
- Постоянный мониторинг долгосрочного эффекта

- Отслеживать, не снижается ли эффективность бонуса со временем.

- Смотреть на LTV когорт, пришедших по бонусу, через 30/60/90 дней.

- Оценить, какой размер бонуса даёт наилучшее соотношение «прирост конверсии / потеря маржи».

- Если бонус работает на TikTok, протестировать его на других каналах с низкой конверсией (например, Affiliate, Display).
